# Milestone 1 — scRNA-seq receptor heatmaps (WMB-10Xv3)

Mean log2(CPM+1) per **cell type** × **brain area**, driven entirely by `receptor_query_config.yaml`.

Requires internet on first run for `AbcProjectCache` downloads.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import pandas as pd

from src.config import load_config, get_figures_dir, get_parquet_path
from src.data_loaders import (
    get_abc_cache,
    load_scrna_cell_metadata,
    load_expression_subset,
    aggregate_scrna_expression,
    family_gene_region_matrix,
    combined_heatmap_matrix,
)
from src.utils import build_brain_area_mapping, top_variable_cell_types
from src.plotting import plot_family_heatmap, plot_combined_heatmap

In [2]:
# Figure output directory (outside repo — edit path as needed)
OUTPUT_DIR = Path("/Users/rancze/Documents/Projects/Ach_NE_Marius_Felix/exploration")

CONFIG_PATH = PROJECT_ROOT / "receptor_query_config.yaml"
config = load_config(CONFIG_PATH)
BASE_DIR = PROJECT_ROOT
figures_dir = get_figures_dir(config, output_dir=OUTPUT_DIR)

print(f"Genes: {len(config['_all_genes'])}")
print(f"Brain areas: {config['brain_areas']}")
print(f"Cell type level: {config['cell_type_level']}")
print(f"Figures dir: {figures_dir}")

Genes: 1
Brain areas: ['VISp']
Cell type level: supertype


In [3]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

Manifest: releases/20260415/manifest.json


In [4]:
cell_meta = load_scrna_cell_metadata(cache, config)
print(f"Filtered cells: {len(cell_meta):,}")
print("\nCells per brain_area:")
print(cell_meta.groupby("brain_area").size().sort_values(ascending=False))

area_to_rois, _ = build_brain_area_mapping(cache, config["brain_areas"])
print("\nROI mapping (sample):")
for ba, rois in area_to_rois.items():
    print(f"  {ba}: {sorted(rois)[:8]}{'...' if len(rois) > 8 else ''}")

/Users/rancze/Documents/GitHub/Expresso/src/data_loaders.py:62: UserWarning: brain_area 'VISp' maps to WMB-10X dissection ROI 'VIS' (scRNA-seq has no CCF parcellation at this resolution).
  area_to_rois, assign_brain_area = build_brain_area_mapping(cache, brain_areas)


Filtered cells: 30,882

Cells per brain_area:
brain_area
VISp    30882
dtype: int64

ROI mapping (sample):
  VISp: ['VIS']


/var/folders/n7/dvksxsf55hxbzrrfm5gv88cwmxqfb7/T/ipykernel_99570/268077318.py:6: UserWarning: brain_area 'VISp' maps to WMB-10X dissection ROI 'VIS' (scRNA-seq has no CCF parcellation at this resolution).
  area_to_rois, _ = build_brain_area_mapping(cache, config["brain_areas"])


In [5]:
genes = config["_all_genes"]
adata = load_expression_subset(cache, genes, cell_meta, config)
if adata is None:
    raise RuntimeError("No expression data loaded; check gene names and cache downloads.")
print(adata)

WMB-10Xv3-Isocortex-1-log2.h5ad: 100%|██████████| 11.8G/11.8G [10:20<00:00, 19.0MMB/s]
WMB-10Xv3-Isocortex-2-log2.h5ad: 100%|██████████| 8.36G/8.36G [08:20<00:00, 16.7MMB/s]
WMB-10Xv3 packages: 100%|██████████| 2/2 [19:02<00:00, 571.16s/it]

AnnData object with n_obs × n_vars = 30882 × 1
    obs: 'cell_barcode', 'library_label', 'anatomical_division_label'
    var: 'gene_symbol'


In [6]:
agg_long = aggregate_scrna_expression(adata, cell_meta, config)
print(agg_long.head())
print(f"\nAggregated rows: {len(agg_long):,}")

                      cell_type brain_area   gene  mean_expression     family
0  0001 CLA-EPd-CTX Car3 Glut_1       VISp  Htr2a         7.268156  serotonin
1         0003 IT EP-CLA Glut_1       VISp  Htr2a         6.930685  serotonin
2         0013 L6 IT CTX Glut_1       VISp  Htr2a         0.579868  serotonin
3         0014 L6 IT CTX Glut_2       VISp  Htr2a         2.708646  serotonin
4         0015 L6 IT CTX Glut_3       VISp  Htr2a         0.289879  serotonin

Aggregated rows: 102


In [7]:
print(f"Saving figures to {figures_dir}")

for family in config["_families"]:
    mat = family_gene_region_matrix(agg_long, family, config["brain_areas"])
    if mat.empty:
        warnings.warn(f"No data for family {family!r}; skipping heatmap.")
        continue
    path = plot_family_heatmap(family, mat, config, output_dir=OUTPUT_DIR)
    print(f"Saved {path}")

Saving figures to /Users/rancze/Documents/GitHub/Expresso/figures
Saved /Users/rancze/Documents/GitHub/Expresso/figures/heatmap_serotonin.png


In [8]:
# Combined: all genes × top-50 most variable cell types
gene_ct = agg_long.pivot_table(
    index="cell_type",
    columns="gene",
    values="mean_expression",
    aggfunc="mean",
)
top_ct = top_variable_cell_types(gene_ct, n=50)
combined = combined_heatmap_matrix(agg_long, top_ct, config["brain_areas"])
path = plot_combined_heatmap(combined, config, output_dir=OUTPUT_DIR)
print(f"Saved {path}")

Saved /Users/rancze/Documents/GitHub/Expresso/figures/heatmap_combined.png


In [9]:
if config["output"].get("save_processed_data", True):
    parquet_path = get_parquet_path(config, BASE_DIR)
    agg_long.to_parquet(parquet_path, index=False)
    print(f"Saved aggregated matrix to {parquet_path}")

Saved aggregated matrix to /Users/rancze/Documents/GitHub/Expresso/data/aggregated_scrna.parquet


---
## Dev / smoke test (2 genes × 2 regions)

Run this cell only to validate the pipeline with a small download footprint.

In [ ]:
# Override config for quick test
test_config = load_config(CONFIG_PATH)
test_config["brain_areas"] = ["STR", "TH"]
test_config["receptors"] = {"dopamine": ["Drd1", "Drd2"]}
genes_map = {}
for fam, glist in test_config["receptors"].items():
    for g in glist:
        genes_map[g] = fam
test_config["_genes_flat"] = genes_map
test_config["_all_genes"] = list(genes_map)
test_config["_families"] = list(test_config["receptors"].keys())

test_cache = get_abc_cache(test_config)
test_meta = load_scrna_cell_metadata(test_cache, test_config)
print(f"Test cells: {len(test_meta):,}")
test_adata = load_expression_subset(test_cache, test_config["_all_genes"], test_meta, test_config)
test_agg = aggregate_scrna_expression(test_adata, test_meta, test_config)
test_mat = family_gene_region_matrix(test_agg, "dopamine", test_config["brain_areas"])
plot_family_heatmap("dopamine", test_mat, test_config, output_dir=OUTPUT_DIR)
print(f"Test heatmap saved to {figures_dir / 'heatmap_dopamine.png'}")